In [1]:

import pandas as pd
import random
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')
nltk.download('wordnet')


sports = [
("India Wins Cricket Series","India defeated Australia by six wickets in the final ODI after excellent batting and bowling."),
("Football Team Wins Championship","The football club scored two late goals to secure the league title."),
("Olympic Gold Medal","The athlete won a gold medal after breaking the national record."),
("Tennis Champion","The player lifted the Grand Slam trophy after an exciting final."),
("Kabaddi Tournament","The home team dominated the national kabaddi championship.")
]

politics = [
("New Tax Policy","The government announced a new tax policy to improve economic growth."),
("Election Campaign","Political leaders addressed thousands of supporters during the campaign."),
("Parliament Session","Members debated the new education bill in parliament."),
("Budget Announcement","The finance minister presented the annual budget."),
("International Summit","World leaders discussed climate change and trade policies.")
]

technology = [
("AI Startup Launches Product","A startup introduced an AI assistant for education and healthcare."),
("New Smartphone Released","The company launched a smartphone with advanced AI features."),
("Cyber Security","Experts warned about increasing cyber attacks across industries."),
("Cloud Computing","Businesses are adopting cloud technology to improve efficiency."),
("Software Update","The latest software update improves performance and security.")
]

business = [
("Stock Market Rises","The stock market reached a record high after strong earnings."),
("Company Expansion","The retail company announced expansion into international markets."),
("Startup Funding","The startup raised millions from investors."),
("Bank Profit","The bank reported higher quarterly profits."),
("Electric Vehicle Industry","Automobile companies increased investments in electric vehicles.")
]

entertainment = [
("Movie Breaks Records","The latest movie collected huge revenue worldwide."),
("Music Awards","Popular singers won multiple awards at the annual ceremony."),
("New Web Series","The streaming platform released a successful web series."),
("Film Festival","International filmmakers showcased their latest productions."),
("Celebrity Interview","The actor discussed upcoming projects during an interview.")
]

categories = {
    "Sports": sports,
    "Politics": politics,
    "Technology": technology,
    "Business": business,
    "Entertainment": entertainment
}

rows = []

article_id = 1

for category, articles in categories.items():
    for i in range(200):      # 200 articles per category

        title, article = random.choice(articles)

        rows.append({
            "Article_ID": article_id,
            "Title": title,
            "Article": article,
            "Category": category
        })

        article_id += 1

df = pd.DataFrame(rows)

print("Dataset Shape:", df.shape)
print(df.head())



df["Text"] = df["Title"] + " " + df["Article"]


stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z ]',' ',text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

df["Clean_Text"] = df["Text"].apply(preprocess)

X = df["Clean_Text"]
y = df["Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", MultinomialNB())
])

model.fit(X_train, y_train)



pred = model.predict(X_test)



print("\nAccuracy:", accuracy_score(y_test, pred))

print("\nClassification Report\n")
print(classification_report(y_test, pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, pred))



sample = """
The Indian cricket team defeated England by five wickets
to win the championship after an excellent bowling performance.
"""

sample = preprocess(sample)

prediction = model.predict([sample])

print("\nPredicted Category:", prediction[0])



df.to_csv("news_articles_dataset.csv", index=False)

print("\nDataset saved as news_articles_dataset.csv")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Dataset Shape: (1000, 4)
   Article_ID                            Title  \
0           1               Kabaddi Tournament   
1           2  Football Team Wins Championship   
2           3                  Tennis Champion   
3           4  Football Team Wins Championship   
4           5                  Tennis Champion   

                                             Article Category  
0  The home team dominated the national kabaddi c...   Sports  
1  The football club scored two late goals to sec...   Sports  
2  The player lifted the Grand Slam trophy after ...   Sports  
3  The football club scored two late goals to sec...   Sports  
4  The player lifted the Grand Slam trophy after ...   Sports  

Accuracy: 1.0

Classification Report

               precision    recall  f1-score   support

     Business       1.00      1.00      1.00        40
Entertainment       1.00      1.00      1.00        40
     Politics       1.00      1.00      1.00        40
       Sports       1.00      